# Lab 12 — Convolution, Recurrence and Attention, Built from Scratch

**Class 12.** Guide: `practice/12_neural_networks.md`. Lecture: `slides/12_neural_networks.pdf`.
Demos: `01_mlp_limits.ipynb` to `08_forecast_stack.ipynb`. In class (60 minutes): Parts A to D; at home: Parts E and F.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready:
it draws the plots and prints the comparisons once your functions work.

**The rule of this lab.** Each structure of the class is one small function: a convolution, an LSTM step, scaled dot-product attention.
You write it with NumPy and compare it with its Keras twin **with the same weights**. The notebook prints the difference; explain it.

**Solution notebook.** Before each function: how we got there. After each experiment: the answers, with the numbers of one run
(seeds fixed; times and the last digit of an accuracy can differ on your machine).

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import warnings

import keras
import numpy as np
import polars as pl
from lab12_utils import (
    climate,
    digits_split,
    lorenz,
    memory,
    plot_attention,
    plot_maps,
    random_shift,
    ridge_mae,
    ridge_rmse,
)

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
BLUE, ORANGE, RED = "#0072B2", "#E69F00", "#D55E00"
rng = np.random.default_rng(0)

# Part A — A Convolution by Hand (12 min)

A filter `w` of shape `(F, F, C, C')` slides over an image `x` of shape `(H, W, C)`: the output at `(i, j)` for filter `k` is
$\sum_{a,b,c} w_{a,b,c,k}\, x_{iS+a,\,jS+b,\,c} + b_k$. Zeros are added around the image first (`pad`), the step is `stride`.

## A1. The convolution and the pooling

Implement `conv2d(x, w, b, stride, pad)` (return `(H', W', C')` with $H' = \lfloor (H + 2P - F)/S \rfloor + 1$) and `max_pool(x, size)`
(non-overlapping blocks, stride = size). Use `np.pad` and a double loop over the output positions.

**How we got there.** Pad once with `np.pad(x, ((P, P), (P, P), (0, 0)))`. For each output position cut the patch `xp[iS:iS+F, jS:jS+F, :]` and
take its dot product with each of the `C'` filters: `np.tensordot(patch, w, axes=([0, 1, 2], [0, 1, 2]))` gives all `C'` numbers at once.
Max pooling reshapes the map to `(H/2, 2, W/2, 2, C)` and takes the maximum over the two block axes.

In [ ]:
def conv2d(x, w, b=None, stride=1, pad=0):
    F = w.shape[0]
    xp = np.pad(x, ((pad, pad), (pad, pad), (0, 0)))
    ho = (x.shape[0] + 2 * pad - F) // stride + 1
    wo = (x.shape[1] + 2 * pad - F) // stride + 1
    out = np.zeros((ho, wo, w.shape[3]))
    for i in range(ho):
        for j in range(wo):
            patch = xp[i * stride : i * stride + F, j * stride : j * stride + F, :]
            out[i, j] = np.tensordot(patch, w, axes=([0, 1, 2], [0, 1, 2]))
    return out + (0 if b is None else b)


def max_pool(x, size=2):
    h, w, c = x.shape
    return x[: h // size * size, : w // size * size].reshape(h // size, size, w // size, size, c).max(axis=(1, 3))

The driver takes a digit of the course data and a `Conv2D` layer of Keras (random weights), and compares the two outputs for two settings.

In [ ]:
Xtr, ytr, Xva, yva, Xte, yte = digits_split()
digit = Xtr[0]
rows = []
for stride, pad in [(1, 0), (2, 1)]:
    layer = keras.layers.Conv2D(4, 3, strides=stride, padding="valid")
    zp = keras.layers.ZeroPadding2D(pad) if pad else keras.layers.Identity()
    ours_in = digit
    ref = np.array(layer(zp(digit[None])))[0]
    w_, b_ = layer.get_weights()
    mine = conv2d(ours_in, w_, b_, stride=stride, pad=pad)
    rows.append((stride, pad, mine.shape, ref.shape, f"{np.abs(mine - ref).max():.1e}"))
print(pl.DataFrame(rows, schema=["stride", "pad", "ours", "Keras", "max difference"], orient="row"))
pooled = max_pool(conv2d(digit, w_, b_, 2, 1))
print("pooled map:", pooled.shape)

## A2. Weights and shifts

The cell counts the weights of a convolution and of a dense layer on the same input, and moves the digit one pixel.
Equivariance: moving the digit moves the feature map. A dense layer has no such property.

In [ ]:
n_conv = w_.size + b_.size
n_dense = (64 + 1) * 64
print(f"conv 4 filters 3x3x1: {n_conv} weights; dense with 64 units on the 8x8 digit: {n_dense}")
moved = np.roll(digit, 1, axis=1)
a, b = conv2d(digit, w_, b_), conv2d(moved, w_, b_)
inner = np.abs(a[:, 1:-1] - np.roll(b, -1, axis=1)[:, 1:-1]).max()
print(f"feature map of the moved digit vs the moved feature map (interior): {inner:.1e}")
plot_maps(digit, [conv2d(digit, w_, b_)[..., k] for k in range(4)])

**Questions A.**

1. Your output and Keras differ by about $10^{-7}$. Why not exactly 0?
2. Stride 2 and padding 1 on an $8 \times 8$ digit give what size? Check with the formula.
3. Which of the weights counted above does not depend on the image size, and which does?

**Answers A** (this run).

1. Keras computes in 32-bit floats and the order of the additions differs: the two results agree to rounding, about $10^{-7}$.
2. $\lfloor (8 + 2 - 3)/2 \rfloor + 1 = 4$: the driver prints `(4, 4, 4)`.
3. The filter weights ($4 \cdot 9 + 4 = 40$) do not; the dense layer ($65 \cdot 64 = 4\,160$) grows with the number of pixels.

# Part B — The LSTM and the GRU Step (15 min)

Keras stores the weights as `W` $(d, 4u)$, `U` $(u, 4u)$ and `b` $(4u,)$ with the gates in the order input $i$, forget $f$, candidate $g$, output $o$:
$z = x W + h U + b$ is cut into four blocks of $u$ columns.

## B1. One LSTM step and a run over a sequence

Implement `lstm_step(x, h, c, W, U, b)` returning the new `(h, c)`, and `run_lstm(X, W, U, b)` that loops over the rows of a sequence `X` `(T, d)`
from $h_0 = c_0 = 0$ and returns the matrix of all hidden states `(T, u)`. The sigmoid is given.

**How we got there.** The four gates share one matrix product: `z = x @ W + h @ U + b`, then `z[:u]`, `z[u:2u]`, ... The new cell is
`sigmoid(f) * c + sigmoid(i) * tanh(g)` (forget what is old, write what is new) and the new state `sigmoid(o) * tanh(c)`.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def lstm_step(x, h, c, W, U, b):
    u = h.shape[-1]
    z = x @ W + h @ U + b
    i, f, g, o = (z[k * u : (k + 1) * u] for k in range(4))
    c_new = sigmoid(f) * c + sigmoid(i) * np.tanh(g)
    return sigmoid(o) * np.tanh(c_new), c_new


def run_lstm(X, W, U, b):
    u = U.shape[0]
    h, c, out = np.zeros(u), np.zeros(u), []
    for x in X:
        h, c = lstm_step(x, h, c, W, U, b)
        out.append(h)
    return np.array(out)

The driver builds a Keras `LSTM` with 6 units on a random sequence of 12 steps and 3 inputs, and compares all hidden states.
Then the same for a `GRU` (provided: `gru_step` follows the equations of the lecture).

In [ ]:
def gru_step(x, h, W, U, b):
    u = h.shape[-1]
    zx, zh = x @ W + b[0], h @ U + b[1]
    z = sigmoid(zx[:u] + zh[:u])
    r = sigmoid(zx[u : 2 * u] + zh[u : 2 * u])
    cand = np.tanh(zx[2 * u :] + r * zh[2 * u :])
    return z * h + (1 - z) * cand


seq = rng.normal(size=(12, 3)).astype("float32")
lstm = keras.layers.LSTM(6, return_sequences=True)
lstm.build((None, 12, 3))
W, U, b = lstm.get_weights()
mine = run_lstm(seq, W, U, b)
ref = np.array(lstm(seq[None]))[0]
print(
    "LSTM: parameters",
    W.size + U.size + b.size,
    "= 4u(d + u + 1) =",
    4 * 6 * (3 + 6 + 1),
    f"; max difference {np.abs(mine - ref).max():.1e}",
)
gru = keras.layers.GRU(6, return_sequences=True)
gru.build((None, 12, 3))
Wg, Ug, bg = gru.get_weights()
h, hs = np.zeros(6), []
for x in seq:
    h = gru_step(x, h, Wg, Ug, bg)
    hs.append(h)
print(
    "GRU : parameters",
    Wg.size + Ug.size + bg.size,
    f"; max difference {np.abs(np.array(hs) - np.array(gru(seq[None]))[0]).max():.1e}",
)

**Questions B.**

1. Set the forget-gate bias to $+5$ and then to $-5$ (the second block of `b`) and print the last cell state of a run. What does each setting do to the memory?
2. The LSTM has 4 weight sets and the GRU 3. Count the parameters of both for $u = 16$, $d = 5$ and compare with the formulas of the lecture.

**Answers B** (this run).

1. With a large positive forget bias $f_t \approx 1$: the cell keeps its old content and adds the new one slowly, so the cell state grows and changes little;
   with a large negative bias $f_t \approx 0$: the old content is erased at each step, the cell is a function of the last input only.
2. LSTM $4 \cdot 16 \cdot (5 + 16 + 1) = 1\,408$; GRU $3\,(16 \cdot 5 + 16 \cdot 16 + 2 \cdot 16) = 1\,104$ (Keras `reset_after=True` has two bias vectors).

# Part C — Scaled Dot-Product Attention (15 min)

$A = \mathrm{softmax}(Q K^\top / \sqrt{d_k})$ row by row, output $AV$. A boolean `mask` of shape `(T, T)` is `True` where a position may be looked at.

## C1. Attention and positions

Implement `attention(Q, K, V, mask=None)` returning `(output, A)`: scores of the forbidden positions are replaced by `-1e9` before the softmax.
Implement `positional_encoding(T, d)`: column $2i$ is $\sin(p/10000^{2i/d})$ and column $2i + 1$ is $\cos(p/10000^{2i/d})$. The softmax is given.

**How we got there.** The scores are one matrix product, `Q @ K.T / np.sqrt(d_k)`. A row-wise softmax needs the maximum subtracted for stability.
The mask only changes the scores before the softmax, so the forbidden weights are exactly 0 and the rows still sum to 1. For the positions,
`np.arange(T)[:, None] / 10000 ** (2 * np.arange(d // 2) / d)` is a `(T, d/2)` array of angles; the sines go to the even columns and the cosines to the odd ones.

In [ ]:
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def attention(Q, K, V, mask=None):
    scores = Q @ K.T / np.sqrt(K.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    A = softmax(scores)
    return A @ V, A


def positional_encoding(T, d):
    ang = np.arange(T)[:, None] / 10000 ** (2 * np.arange(d // 2)[None, :] / d)
    pe = np.zeros((T, d))
    pe[:, 0::2], pe[:, 1::2] = np.sin(ang), np.cos(ang)
    return pe

The driver compares your function with one head of `MultiHeadAttention` (same projections), applies a causal mask, and tests the permutation
property with and without positions.

In [ ]:
X = rng.normal(size=(1, 6, 8)).astype("float32")
mha = keras.layers.MultiHeadAttention(1, 4, use_bias=False)
mha(X, X)
wq, wk, wv, wo = mha.get_weights()
Q, K, V = (X[0] @ w[:, 0, :] for w in (wq, wk, wv))
out, A = attention(Q, K, V)
_, kw = mha(X, X, return_attention_scores=True)
print(
    f"output vs Keras: {np.abs(out @ wo[0] - np.array(mha(X, X))[0]).max():.1e}; weights vs Keras: {np.abs(A - np.array(kw)[0, 0]).max():.1e}"
)
causal = np.tril(np.ones((6, 6), dtype=bool))
_, Ac = attention(Q, K, V, causal)
print("rows sum to 1:", np.allclose(Ac.sum(1), 1), "; weights above the diagonal:", float(np.triu(Ac, 1).sum()))
perm = rng.permutation(6)
x0 = X[0]
pe = positional_encoding(6, 8)
o1 = attention(x0, x0, x0)[0]
o2 = attention(x0[perm], x0[perm], x0[perm])[0]
p1 = attention(x0 + pe, x0 + pe, x0 + pe)[0]
p2 = attention(x0[perm] + pe, x0[perm] + pe, x0[perm] + pe)[0]
print(
    f"permute the tokens, no positions: {np.abs(o2 - o1[perm]).max():.1e}; with positions: {np.abs(p2 - p1[perm]).max():.2f}"
)
plot_attention(A, Ac)

**Questions C.**

1. Replace `np.sqrt(K.shape[-1])` by 1 and print the largest entry of each row of $A$ for $d_k = 4$ and $d_k = 256$ (random $Q$, $K$). What changes and why?
2. Without positions the layer returns the permuted rows. Why does adding $PE$ break this property, and why is that what we want?

**Answers C** (this run).

1. Without the scale the scores have standard deviation $\sqrt{d_k}$: for $d_k = 256$ the largest weight of a row is close to 1 (one key takes everything,
   the others get no gradient); with the scale it stays near the uniform value for random inputs.
2. $PE_p$ is added to token $p$, so permuting the tokens changes what each token carries: $A(PX + PE) \ne P\,A(X + PE)$. Order information is now part of the input.

# Part D — Memory against the Lag: RNN, LSTM, GRU, Transformer (15 min)

The first step of a sequence holds one of four symbols, then `lag` steps of noise follow; the label is the symbol (chance 0.25).
`memory(lag, n, seed)` returns `(x, y)` with `x` of shape `(n, lag + 1, 5)`.

## D1. The recurrent model

Implement `build(kind, input_shape, units=32)`: a Keras `Sequential` with an `Input`, one recurrent layer (`SimpleRNN`, `LSTM` or `GRU`, chosen by `kind`)
and a `Dense(4, activation="softmax")`; compile with Adam (learning rate 3e-3, `clipnorm=1.0`) and `sparse_categorical_crossentropy`, metric accuracy.

**How we got there.** The three layers share their interface: a dictionary from the name to the class avoids three copies of the model. The recurrent layer returns
only its last state (default), which is what a many-to-one classifier needs. Clipping is part of the optimizer.

In [ ]:
def build(kind, input_shape, units=32):
    cell = {"rnn": keras.layers.SimpleRNN, "lstm": keras.layers.LSTM, "gru": keras.layers.GRU}[kind]
    keras.utils.set_random_seed(0)
    model = keras.Sequential(
        [keras.layers.Input(input_shape), cell(units), keras.layers.Dense(4, activation="softmax")]
    )
    model.compile(
        optimizer=keras.optimizers.Adam(3e-3, clipnorm=1.0),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

The driver trains each model for 15 epochs at lag 10 and at lag 40 (3 000 training sequences) and prints the test accuracy; the transformer block is built by the helper.

In [ ]:
from lab12_utils import transformer

rows = []
for lag in (10, 40):
    x, y = memory(lag, 3000, seed=1)
    xt, yt = memory(lag, 1000, seed=2)
    row = {"lag": lag}
    for kind in ("rnn", "lstm", "gru", "transformer"):
        m = transformer(x.shape[1:]) if kind == "transformer" else build(kind, x.shape[1:])
        m.fit(x, y, epochs=15, batch_size=64, verbose=0)
        row[kind] = m.evaluate(xt, yt, verbose=0)[1]
    rows.append(row)
print(pl.DataFrame(rows))

**Questions D.**

1. Which models remember at lag 40 and which do not? Link each result to the gradient of its path (a product of Jacobians, a product of gates, one hop).
2. Change `clipnorm=1.0` to `None` for the RNN at lag 10. What happens, and what does clipping fix and not fix?
3. The test sequences use another seed than the training ones. Why does the symbol of a test sequence still count as new data?

**Answers D** (this run: lag 40, RNN 0.243, LSTM 0.997, GRU 0.266, transformer 1.000; all four are at 1.0 for lag 10).

1. The vanilla RNN fails as soon as the lag grows (product of $T$ Jacobians with norm below 1); the LSTM keeps the symbol through the cell path (a product of forget gates);
   the transformer reads step 0 in one hop from the last position, so the lag does not matter.
2. Without clipping an occasional huge gradient can throw the weights away (loss up, accuracy at chance); clipping fixes explosion but not vanishing: a gradient that has
   shrunk to zero is not made larger by clipping.
3. The symbols are random draws from the same four values, but the noise and the position of each symbol in the dataset are new: the model has to generalise, not look up.

# Part E (home) — A CNN against the MLP on Shifted Digits

## E1. The CNN

Implement `build_cnn(filters=(8, 16), gap=False)`: `Conv2D(f, 3, padding="same", activation="relu")` and `MaxPooling2D(2)` for each `f`, then `Flatten()`
(or `GlobalAveragePooling2D()` when `gap`), then `Dense(10, activation="softmax")`; compile with Adam and `sparse_categorical_crossentropy`.

**How we got there.** The input is `(8, 8, 1)`; each block halves the map. With `gap=True` the dense head sees one number per filter and loses the position.

In [ ]:
def build_cnn(filters=(8, 16), gap=False):
    keras.utils.set_random_seed(0)
    layers = [keras.layers.Input((8, 8, 1))]
    for f in filters:
        layers += [keras.layers.Conv2D(f, 3, padding="same", activation="relu"), keras.layers.MaxPooling2D(2)]
    layers.append(keras.layers.GlobalAveragePooling2D() if gap else keras.layers.Flatten())
    layers.append(keras.layers.Dense(10, activation="softmax"))
    model = keras.Sequential(layers)
    model.compile(optimizer=keras.optimizers.Adam(3e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

The driver trains the MLP, your CNN and the CNN with a global average pool, with and without three shifted copies of the training digits, and tests on digits shifted by up to 0, 1 and 2 pixels.

In [ ]:
def mlp():
    keras.utils.set_random_seed(0)
    model = keras.Sequential(
        [
            keras.layers.Input((8, 8, 1)),
            keras.layers.Flatten(),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(32, activation="relu"),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(optimizer=keras.optimizers.Adam(3e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


tests = {f"s={s}": (random_shift(Xte, s, seed=1), yte) for s in (0, 1, 2)}
Xaug = np.concatenate([Xtr] + [random_shift(Xtr, 2, seed=10 + i) for i in range(3)])
yaug = np.concatenate([ytr] * 4)
rows = []
for name, make in {
    "MLP": mlp,
    "CNN": lambda: build_cnn((8, 16)),
    "CNN + global pool": lambda: build_cnn((16, 32), gap=True),
}.items():
    for aug, (xa, ya) in {"plain": (Xtr, ytr), "augmented": (Xaug, yaug)}.items():
        m = make()
        m.fit(xa, ya, epochs=20, batch_size=32, verbose=0)
        rows.append(
            {
                "model": name,
                "training": aug,
                "weights": m.count_params(),
                **{k: m.evaluate(*v, verbose=0)[1] for k, v in tests.items()},
            }
        )
print(pl.DataFrame(rows))

**Questions E.**

1. With centred digits the CNN is not much better than the MLP. Where does the structure pay off, and by how much?
2. Why does the global average pool help more than the dense head after `Flatten`?
3. Augmentation helps the MLP too. What does that say about where the gain of the CNN comes from?

**Answers E** (this run, plain training, shift 1 / shift 2: MLP 0.386 / 0.203, CNN 0.572 / 0.275, CNN + global pool 0.617 / 0.381; centred digits 0.97 for all).

1. On shifted digits: the CNN loses less than the MLP, and the CNN with a global pool the least; the CNN also has 1 898 weights against 6 570.
2. A dense head after `Flatten` has one weight per position of each feature, so it reads *where*; the global pool keeps one number per filter and throws the position away.
3. With augmentation (shift 2) the MLP reaches 0.875, the CNN 0.911 and the CNN with the pool 0.933. Augmentation puts the invariance into the data instead of the architecture: the MLP can learn it with its weights, the CNN gets it for free from weight sharing and needs the data less.

# Part F (home) — Challenge: Forecast a Room Temperature

`climate(horizon=6)` returns the windows of Demo 12e: `xtr, ytr, xva, yva, xte, yte, sd` (24 hours of five temperatures, the change in room 1 after `horizon` hours, the last 15 % as test,
`sd` converts to degrees). The driver prints the baselines. Implement `forecast_mae(kind, seed)`: train a model of your choice
(`"rnn"`, `"lstm"` or `"gru"` with 16 units, or your own) with early stopping on the validation part and return the test MAE in °C.

**How we got there.** The target is the change from the last observed value (the persistence forecast is 0), scaled by its training standard deviation; the early stopping restores the best epoch on the validation part.

In [ ]:
xtr, ytr, xva, yva, xte, yte, sd = climate(horizon=6)
scale = ytr.std()


def forecast_mae(kind, seed=0):
    cell = {"rnn": keras.layers.SimpleRNN, "lstm": keras.layers.LSTM, "gru": keras.layers.GRU}[kind]
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([keras.layers.Input(xtr.shape[1:]), cell(16), keras.layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(3e-3, clipnorm=1.0), loss="mse")
    m.fit(
        xtr,
        ytr / scale,
        epochs=60,
        batch_size=64,
        validation_data=(xva, yva / scale),
        verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)],
    )
    return float(np.abs(m.predict(xte, verbose=0).ravel() * scale - yte).mean() * sd)

In [ ]:
rows = [("persistence", float(np.abs(yte).mean() * sd)), ("ridge on the window", ridge_mae(xtr, ytr, xte, yte, sd))]
for kind in ("rnn", "lstm", "gru"):
    maes = [forecast_mae(kind, s) for s in range(3)]
    rows.append((f"{kind.upper()} 16 units", float(np.mean(maes))))
print(pl.DataFrame(rows, schema=["model", "test MAE (C)"], orient="row"))

**Questions F.**

1. Does any recurrent model beat persistence and the ridge regression? Report the mean and the spread over seeds, not the best seed.
2. Which change of the protocol (features, window, target, regularisation) would you try first, and why?

**Answers F** (this run, test MAE in °C, mean of 3 seeds: persistence 0.450, ridge 0.358, RNN 0.422, LSTM 0.854, GRU 0.909).

1. On 2 300 training windows with a shifted test period the linear model on the window is hard to beat; a recurrent model that does is within the spread of its seeds, so one seed proves nothing.
2. Features that carry the cycle (hour of the day), a smaller or regularised network, and a target the model cannot get wrong (the change from the last value, as here) come before a bigger network.

## F2. A nonlinear series and a frozen stack

On the Lorenz series (`lorenz()`: noisy, nonlinear, ten steps ahead) a linear model fails. `fit_regressor(kind, seed)` is ready: it trains one recurrent regressor (32 units) and returns it.
Implement `stack_rmse(models)`: take the trained regressors, cut off their last `Dense` layer (`keras.Model(m.inputs, m.layers[-2].output)`), freeze them (`trainable = False`), concatenate their features, train a `Dense(32, relu)` and a `Dense(1)` on top (early stopping on the validation part) and return the test RMSE.

In [ ]:
ltr, lytr, lva, lyva, lte, lyte = lorenz()


def fit_regressor(kind, seed=0):
    cell = {"rnn": keras.layers.SimpleRNN, "lstm": keras.layers.LSTM, "gru": keras.layers.GRU}[kind]
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([keras.layers.Input(ltr.shape[1:]), cell(32), keras.layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(3e-3, clipnorm=1.0), loss="mse")
    m.fit(
        ltr,
        lytr,
        epochs=30,
        batch_size=64,
        validation_data=(lva, lyva),
        verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    )
    return m


def rmse(m):
    return float(np.sqrt(np.mean((np.ravel(m.predict(lte, verbose=0)) - lyte) ** 2)))


def stack_rmse(models):
    feats = []
    for m in models:
        enc = keras.Model(m.inputs, m.layers[-2].output)
        enc.trainable = False
        feats.append(enc(models[0].inputs[0]))
    keras.utils.set_random_seed(0)
    x = keras.layers.Dense(32, activation="relu")(keras.layers.Concatenate()(feats))
    head = keras.Model(models[0].inputs, keras.layers.Dense(1)(x))
    head.compile(optimizer=keras.optimizers.Adam(3e-3), loss="mse")
    head.fit(
        ltr,
        lytr,
        epochs=30,
        batch_size=64,
        validation_data=(lva, lyva),
        verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    )
    return rmse(head)

In [ ]:
trio = {kind: fit_regressor(kind) for kind in ("rnn", "lstm", "gru")}
rows2 = [
    ("persistence", float(np.sqrt(np.mean((lte[:, -1, 0] - lyte) ** 2)))),
    ("ridge on the window", ridge_rmse(ltr, lytr, lte, lyte)),
]
rows2 += [(kind.upper() + ", 32 units", rmse(m)) for kind, m in trio.items()]

In [ ]:
rows2.append(("frozen encoders + MLP head", stack_rmse(list(trio.values()))))
print(pl.DataFrame(rows2, schema=["model", "test RMSE"], orient="row"))

**Questions F2.**

1. Ridge won on the climate series and loses on the Lorenz series. What property of the series explains it?
2. Does the head beat the best single recurrent layer? Why is it cheap to train?
3. What would change if the encoders were not frozen, and what would you have to watch?

**Answers F2** (this run, one seed, test RMSE: persistence 1.34, ridge 0.99, the recurrent layers 0.38 to 0.43, the stack about 0.34).

1. The Lorenz series is nonlinear: the next value depends on the window through products and sign flips that a weighted sum of the 24 values cannot express. The room temperature is smooth and almost linear in its past.
2. Yes, by about 10 percent: the three encoders see the window in three ways and the head combines their features. Only 3 137 weights are trained, on features that are computed once.
3. The whole model (11 937 weights) would be trained again on the same windows: it can fit the training part better and overfit it; keep the early stopping on the validation part and compare with the frozen version.